# KofamScan CTS Demo

End-to-end test of the cdm_kofamscan CTS tool. Re-executed 2026-11-06 after the tool audit (7-column parse, assigned-KO count, tmp dir outside `/out`).

- **Image:** `ghcr.io/kbaseincubator/cdm_kofamscan:0.1.0@sha256:d6b20eccf4c6bf1b095e530844a8b04dbae5fca85daf0c9b2bdffb0cf10a9a42`
- **Refdata UUID:** `84b31af0-a5a7-4016-906c-9ad9eef34c6a`
- **Refdata file:** `cts-refdata/kofam/2025-04-30/kofam_refdata.tar.gz`
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/kofamscan/demo_20261106`

KofamScan annotates protein sequences with KEGG KO terms using HMM profiles. Input must be **protein FASTA**, not nucleotide.

## 1. Setup

In [1]:
tscli = get_task_service_client()
mincli = get_minio_client()

IMAGE = "ghcr.io/kbaseincubator/cdm_kofamscan:0.1.0@sha256:d6b20eccf4c6bf1b095e530844a8b04dbae5fca85daf0c9b2bdffb0cf10a9a42"
OUTPUT_DIR = "cts/io/jplfaria/output/kofamscan/demo_20261106"

print(tscli.whoami())

{'user': 'jplfaria', 'roles': ['cts_user'], 'allowed_paths': [{'path': 'cts/io/', 'perm': 'write'}]}


## 2. Find protein FASTA inputs

KofamScan needs proteins (.faa). There are protein files left over from a checkm2 demo run in `cts/io/gavin/test_output/kb_demo/`. We use those as a quick test input.

In [2]:
candidates = []
for o in mincli.list_objects("cts", prefix="io/gavin/test_output/kb_demo", recursive=True):
    if o.object_name.endswith(".faa"):
        candidates.append(f"cts/{o.object_name}")

print(f"Found {len(candidates)} protein FASTA file(s):")
for f in candidates:
    print(f"  {f}")

input_files = candidates

Found 4 protein FASTA file(s):
  cts/io/gavin/test_output/kb_demo/0/protein_files/GCA_000008085.1_ASM808v1_genomic.faa
  cts/io/gavin/test_output/kb_demo/0/protein_files/GCA_000010565.1_ASM1056v1_genomic.faa
  cts/io/gavin/test_output/kb_demo/1/protein_files/GCA_000145985.1_ASM14598v1_genomic.faa
  cts/io/gavin/test_output/kb_demo/1/protein_files/GCA_000147015.1_ASM14701v1_genomic.faa


## 3. Submit KofamScan job

We do a quick run with one container per input file. KofamScan with all 28K KEGG profiles can be slow on large inputs; a small bacterial proteome should finish in 5-15 min per genome.

`exec_annotation` takes exactly one query file, so `num_containers=len(input_files)`. The tmp dir is `/tmp/kofam`, outside the output mount; under `/out` its scratch files (26-48 MB per container) would be uploaded with the results.

In [3]:
if not input_files:
    raise RuntimeError("No protein FASTA inputs found. Adjust the prefix above or upload .faa files first.")

job = tscli.submit_job(
    IMAGE,
    input_files,
    OUTPUT_DIR,
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "-p", "/ref_data/profiles",
        "-k", "/ref_data/ko_list",
        "-o", "/out/annotations.tsv",
        "-f", "detail-tsv",
        "--cpu", "4",
        "--tmp-dir", "/tmp/kofam",     # outside /out, so scratch is not uploaded
        tscli.insert_files(),
    ],
    num_containers=len(input_files),
    cpus=4,
    memory="16GB",
    runtime="PT2H",
)
print("Job ID:", job.id)

Job ID: 3cd80e30-36c8-4f55-90a9-f2d44589e31d


## 4. Wait for completion

In [4]:
# Block until the job finishes (CTS keeps container logs only for failed jobs,
# so the checks below validate the output files themselves).
import json
job.wait_for_completion()
status = job.get_job_status()
print("state:", status["state"])
print("exit codes:", job.get_exit_codes().get("exit_codes"))


state: complete
exit codes: [0, 0, 0, 0]


## 5. Inspect output files

In [5]:
outs = job.get_job()["outputs"]
for o in outs:
    print(o["crc64nvme"], o["file"])
print(f"\n{len(outs)} output files; scratch files uploaded: {sum('/tmp/' in o['file'] for o in outs)}")


n2mQ8pUGvpc= cts/io/jplfaria/output/kofamscan/demo_20261106/0/annotations.tsv
gANvIuZ5Jk4= cts/io/jplfaria/output/kofamscan/demo_20261106/1/annotations.tsv
ybhogMK9Lk0= cts/io/jplfaria/output/kofamscan/demo_20261106/2/annotations.tsv
fIVcDnaTmEE= cts/io/jplfaria/output/kofamscan/demo_20261106/3/annotations.tsv

4 output files; scratch files uploaded: 0


## 6. Read annotations.tsv as a dataframe

`detail-tsv` has **7** tab-separated columns: `significance`, `gene_name`, `KO`, `threshold`, `score`, `e_value`, `KO_definition`.
The first column is `*` when the hit scores above the KO's adaptive threshold. **Only `*` rows are KO assignments**; the other rows are sub-threshold hits.
Lines starting with `#` are header comments.

In [6]:
import io
import pandas as pd

outs_tsv = [o for o in outs if o["file"].endswith("annotations.tsv")]
print(f"{len(outs_tsv)} annotations.tsv file(s), one per input")

cols = ["significance", "gene_name", "KO", "threshold", "score", "e_value", "KO_definition"]
frames = []
for o in outs_tsv:
    bucket, key = o["file"].split("/", 1)
    raw = mincli.get_object(bucket, key).read().decode("utf-8")
    df = pd.read_csv(io.StringIO(raw), sep="\t", comment="#", header=None, names=cols, dtype=str)
    df["source_file"] = o["file"]
    frames.append(df)

all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=cols)
assigned = all_df[all_df["significance"].fillna("").str.strip() == "*"]
print(f"HMM hits (all rows):        {len(all_df):,}")
print(f"KO assignments (* rows):    {len(assigned):,}")
print(f"genes with an assigned KO:  {assigned['gene_name'].nunique():,}")
print(f"distinct assigned KOs:      {assigned['KO'].nunique():,}")
print("\nper input file:")
print(all_df.assign(assigned=all_df["significance"].fillna("").str.strip() == "*")
      .groupby("source_file")["assigned"].agg(hits="size", assigned="sum").to_string())
assigned.head(10)


4 annotations.tsv file(s), one per input


HMM hits (all rows):        147,510
KO assignments (* rows):    2,769
genes with an assigned KO:  2,639
distinct assigned KOs:      1,711

per input file:
                                                                   hits  assigned
source_file                                                                      
cts/io/jplfaria/output/kofamscan/demo_20261106/0/annotations.tsv  10662       218
cts/io/jplfaria/output/kofamscan/demo_20261106/1/annotations.tsv  84965      1716
cts/io/jplfaria/output/kofamscan/demo_20261106/2/annotations.tsv  49228       713
cts/io/jplfaria/output/kofamscan/demo_20261106/3/annotations.tsv   2655       122


,significance,gene_name,KO,threshold,score,e_value,KO_definition,source_file
0,*,AE017199.1_1,K19091,38.33,106.1,1.2e-31,CRISPR-associated endoribonuclease Cas6 [EC:3....,cts/io/jplfaria/output/kofamscan/demo_20261106...
2,*,AE017199.1_2,K03724,281.63,744.2,1.2e-224,ATP-dependent helicase Lhr and Lhr-like helica...,cts/io/jplfaria/output/kofamscan/demo_20261106...
107,*,AE017199.1_4,K24956,20.63,120.1,3.2e-36,protein archease,cts/io/jplfaria/output/kofamscan/demo_20261106...
108,*,AE017199.1_5,K15566,166.90,276.7,7.4e-84,tRNA (adenine9-N1/guanine9-N1)-methyltransfera...,cts/io/jplfaria/output/kofamscan/demo_20261106...
119,*,AE017199.1_8,K02922,43.70,48.7,2.4e-14,large subunit ribosomal protein L37e,cts/io/jplfaria/output/kofamscan/demo_20261106...
125,*,AE017199.1_9,K15865,396.87,418.5,1.4e-126,threonylcarbamoyladenosine tRNA methylthiotran...,cts/io/jplfaria/output/kofamscan/demo_20261106...
143,*,AE017199.1_12,K05919,47.73,96.6,4.7e-29,superoxide reductase [EC:1.15.1.2],cts/io/jplfaria/output/kofamscan/demo_20261106...
200,*,AE017199.1_15,K03304,243.80,305.1,2.3e-92,tellurite resistance protein,cts/io/jplfaria/output/kofamscan/demo_20261106...
206,*,AE017199.1_16,K03465,111.73,152.3,6.4e-46,thymidylate synthase (FAD) [EC:2.1.1.148],cts/io/jplfaria/output/kofamscan/demo_20261106...
230,*,AE017199.1_19,K15342,55.50,218.0,9e-66,CRISP-associated protein Cas1,cts/io/jplfaria/output/kofamscan/demo_20261106...


## 7. End-to-end check

- Job state is `complete` and every container exited 0
- One `annotations.tsv` per input file
- At least one KO assignment (`*` row) per file, with KEGG KO IDs
- No scratch files uploaded

In [7]:
per_file = assigned.groupby("source_file").size()
checks = {
    "job complete, all exit 0": job.get_job_status()["state"] == "complete"
                                and set(job.get_exit_codes().get("exit_codes", [])) == {0},
    "one annotations.tsv per input": len(outs_tsv) == len(input_files),
    "every file has KO assignments": len(per_file) == len(outs_tsv) and (per_file > 0).all(),
    "assigned KOs look like K#####": assigned["KO"].str.match(r"^K\d{5}$").all() if len(assigned) else False,
    "no scratch files uploaded": not any("/tmp/" in o["file"] for o in outs),
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
print("\nAll green." if all(checks.values()) else "\nSomething's off, inspect above.")


  [x] job complete, all exit 0
  [x] one annotations.tsv per input
  [x] every file has KO assignments
  [x] assigned KOs look like K#####
  [x] no scratch files uploaded

All green.
